# Class-Aware Token Pruning — DynamicViT with Imbalance-Aware Ratio Loss

Trains three DynamicViT models (keep ratios 70 / 50 / 30 %) using
`class_aware_ratio_loss` instead of the standard `ratio_loss`.

**Motivation**: standard ratio_loss assigns the same pruning target to every
sample.  For Pneumonia (1.5%) and Fracture (5.5%), the classification gradient
is dominated by majority classes, so the token predictor never learns to
preserve their diagnostic patches.  The class-aware loss relaxes the per-sample
keeping target proportionally to the rarity of the rarest positive label — 
samples with rare pathologies are allowed to retain more tokens.

**Key reference**: Zhao et al. (MICCAI 2023) show that pruning
disproportionately degrades minority-class detection in long-tailed multi-label
chest X-ray classifiers even when macro AUC is unchanged.

**Runtime on L4 GPU**: ~3 hours per ratio → ~9 hours total.

**Outputs**
- `checkpoints/dynvit_ca_{ratio}pct_best.pt` — CA-trained model per ratio
- `maps/token_masks/masks_ca_{ratio}pct.npy` — [202, 14, 14] binary masks
- `checkpoints/ca_vs_standard_auc.csv` — per-label AUC: CA vs standard DynamicViT
- `checkpoints/ca_vs_standard_iou.csv` — per-label IoU comparison
- `checkpoints/three_way_summary.csv` — head pruning vs DynamicViT vs CA-DynamicViT

In [ ]:
# ── Cell 1 — Environment ──────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/dizertatie_project'):
        !git clone https://github.com/daviDpaD18/diz.git /content/diz
        os.symlink('/content/diz/dizertatie_project', '/content/dizertatie_project')
    sys.path.insert(0, '/content/dizertatie_project')
except ImportError:
    IS_COLAB = False
    sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

print('Colab:', IS_COLAB)

In [ ]:
# ── Cell 2 — Data to local SSD ────────────────────────────────────────────────
if IS_COLAB:
    DRIVE = '/content/drive/MyDrive/dizertatie'
    if not os.path.exists('/content/train'):
        print('Copying train.zip ...')
        !cp {DRIVE}/train.zip /content/train.zip
        !unzip -q /content/train.zip -d /content/
        !rm /content/train.zip
        print('train/ ready.')
    if not os.path.exists('/content/valid'):
        !cp -r {DRIVE}/valid /content/valid
        print('valid/ ready.')
    import importlib, src.config
    importlib.reload(src.config)

!pip install -q grad-cam

import json, numpy as np, pandas as pd, warnings
import torch, torch.nn.functional as F, timm
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import roc_auc_score
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from src.config import IMAGE_ROOT, SPLITS_DIR, WEIGHTS_DIR, CKPT_DIR
from src.dataset import LABEL_COLS, CheXpertDataset, train_transforms, eval_transforms
from src.dynamic_vit import DynamicViT, class_aware_ratio_loss

CKPT_DIR.mkdir(parents=True, exist_ok=True)
MAPS_DIR = (CKPT_DIR.parent / 'maps') if IS_COLAB \
           else Path('..').resolve() / 'maps'
(MAPS_DIR / 'token_masks').mkdir(parents=True, exist_ok=True)
(MAPS_DIR / 'gradcam').mkdir(parents=True, exist_ok=True)

DEVICE = (
    torch.device('cuda')  if torch.cuda.is_available()  else
    torch.device('mps')   if torch.backends.mps.is_available() else
    torch.device('cpu')
)
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_available() else None
print(f'Device: {DEVICE}  AMP: {AMP_DTYPE}')

In [ ]:
# ── Cell 3 — Constants ────────────────────────────────────────────────────────
KEEP_RATIOS  = [0.70, 0.50, 0.30]
TRAIN_EPOCHS = 10 if IS_COLAB else 1
BATCH_SIZE   = 32 if IS_COLAB else 8
NUM_WORKERS  = 4  if IS_COLAB else 0
LR           = 1e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP    = 1.0
RATIO_LAM    = 2.0
RELAX        = 0.3

RARE_LABELS    = ['Pneumonia', 'Fracture', 'Lung Lesion', 'Pleural Other']
BASELINE_SEEDS = [42, 123, 456]

print('Baseline checkpoints:')
for s in BASELINE_SEEDS:
    d = torch.load(CKPT_DIR / f'baseline_seed{s}_best.pt',
                   map_location='cpu', weights_only=False)
    print(f'  seed {s}: AUC {d["val_macro_auc"]:.4f}')

print(f'\nTotal runs: {len(BASELINE_SEEDS)} seeds × {len(KEEP_RATIOS)} ratios = '
      f'{len(BASELINE_SEEDS)*len(KEEP_RATIOS)} checkpoints')

with open(WEIGHTS_DIR / 'class_weights.json') as f:
    cw = json.load(f)
CLASS_WEIGHTS = torch.tensor([cw[l] for l in LABEL_COLS], dtype=torch.float32).to(DEVICE)

def weighted_bce(logits, labels):
    return (F.binary_cross_entropy_with_logits(
        logits, labels, reduction='none') * CLASS_WEIGHTS).mean()

print('Rarest label weights:')
for l in RARE_LABELS:
    print(f'  {l}: {cw.get(l, "N/A")}')

In [ ]:
# ── Cell 4 — Data loaders ─────────────────────────────────────────────────────
TRAIN_CSV = SPLITS_DIR / ('train_full.csv'   if IS_COLAB else 'train_dev5k.csv')
VALID_CSV = SPLITS_DIR / 'valid_frontal.csv'

df_train = pd.read_csv(TRAIN_CSV)
df_valid = pd.read_csv(VALID_CSV)

train_ds     = CheXpertDataset(df_train, IMAGE_ROOT, train_transforms)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=IS_COLAB, drop_last=True)
val_ds       = CheXpertDataset(df_valid, IMAGE_ROOT, eval_transforms)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=IS_COLAB)
val_loader_1 = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

print(f'Train: {len(df_train):,}  Val: {len(df_valid)}')

In [ ]:
# ── Cell 5 — Validation helper ────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels, *_ in val_loader:
        logits = model(imgs.to(DEVICE))
        all_probs.append(torch.sigmoid(logits).cpu())
        all_labels.append(labels)
    probs  = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    auc = {}
    for i, label in enumerate(LABEL_COLS):
        auc[label] = roc_auc_score(labels[:, i], probs[:, i]) \
                     if labels[:, i].sum() > 0 else float('nan')
    return float(np.nanmean(list(auc.values()))), auc

In [ ]:
# ── Cell 6 — Training loop (3 seeds × 3 keep ratios = 9 checkpoints) ─────────
#
# Same structure as notebook 05 but uses class_aware_ratio_loss.
# Checkpoint naming: dynvit_ca_{pct}pct_seed{s}_best.pt

all_train_results = {}

for baseline_seed in BASELINE_SEEDS:
    baseline_ckpt_path = CKPT_DIR / f'baseline_seed{baseline_seed}_best.pt'
    base_ckpt = torch.load(baseline_ckpt_path, map_location='cpu', weights_only=False)

    for keep_ratio in KEEP_RATIOS:
        pct       = int(keep_ratio * 100)
        save_path = CKPT_DIR / f'dynvit_ca_{pct}pct_seed{baseline_seed}_best.pt'

        if save_path.exists():
            ckpt = torch.load(save_path, map_location='cpu', weights_only=False)
            print(f'[CA {pct}% seed{baseline_seed}] Already done — '
                  f'AUC {ckpt["val_macro_auc"]:.4f}. Skipping.')
            all_train_results[(baseline_seed, pct)] = ckpt['val_macro_auc']
            continue

        print(f'\n{"="*60}')
        print(f'CA TOKEN PRUNING {pct}%  seed {baseline_seed}  '
              f'({int(keep_ratio*196)}/196 tokens kept)')
        print(f'{"="*60}')

        base_model = timm.create_model(base_ckpt['hparams']['model'],
                                       pretrained=False, num_classes=14)
        base_model.load_state_dict(base_ckpt['model_state_dict'])
        model = DynamicViT(base_model, final_keep_ratio=keep_ratio).to(DEVICE)

        predictor_params = list(model.predictors.parameters())
        predictor_ids    = {id(p) for p in predictor_params}
        backbone_params  = [p for p in model.parameters() if id(p) not in predictor_ids]

        optimizer = torch.optim.AdamW([
            {'params': backbone_params,  'lr': LR * 0.1},
            {'params': predictor_params, 'lr': LR},
        ], weight_decay=WEIGHT_DECAY)

        total_steps = TRAIN_EPOCHS * len(train_loader)
        scheduler   = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=int(0.05 * total_steps),
            num_training_steps=total_steps)
        scaler = torch.amp.GradScaler() if torch.cuda.is_available() else None

        best_auc_ca = 0.0

        for epoch in range(1, TRAIN_EPOCHS + 1):
            model.train()
            running_cls = running_rat = 0.0

            for imgs, labels, *_ in tqdm(
                    train_loader,
                    desc=f'[CA {pct}% s{baseline_seed}] Ep {epoch}/{TRAIN_EPOCHS}',
                    leave=False):
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()

                if scaler:
                    with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                        logits, decisions = model(imgs, return_decisions=True)
                        loss_cls = weighted_bce(logits, labels)
                        loss_rat = class_aware_ratio_loss(
                            decisions, labels, CLASS_WEIGHTS, keep_ratio, RATIO_LAM, RELAX)
                        loss = loss_cls + loss_rat
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    scaler.step(optimizer); scaler.update()
                else:
                    logits, decisions = model(imgs, return_decisions=True)
                    loss_cls = weighted_bce(logits, labels)
                    loss_rat = class_aware_ratio_loss(
                        decisions, labels, CLASS_WEIGHTS, keep_ratio, RATIO_LAM, RELAX)
                    loss = loss_cls + loss_rat
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()

                scheduler.step()
                running_cls += loss_cls.item()
                running_rat += loss_rat.item()

            macro, auc_per_label = evaluate(model)
            n    = len(train_loader)
            flag = ' ← best' if macro > best_auc_ca else ''
            print(f'  Ep {epoch:>2} | cls {running_cls/n:.4f} '
                  f'| ratio {running_rat/n:.4f} | AUC {macro:.4f}{flag}')

            if macro > best_auc_ca:
                best_auc_ca = macro
                torch.save({
                    'keep_ratio':        keep_ratio,
                    'relax':             RELAX,
                    'baseline_seed':     baseline_seed,
                    'epoch':             epoch,
                    'model_state_dict':  model.state_dict(),
                    'val_macro_auc':     macro,
                    'val_auc_per_label': auc_per_label,
                    'hparams':           base_ckpt['hparams'],
                }, save_path)

        all_train_results[(baseline_seed, pct)] = best_auc_ca
        print(f'[CA {pct}% seed{baseline_seed}] Done. Best AUC: {best_auc_ca:.4f}')

print('\n=== Summary: best AUC per (seed, keep ratio) ===')
header = '         ' + '  '.join(f'{int(r*100)}%' for r in KEEP_RATIOS)
print(header)
for s in BASELINE_SEEDS:
    row = f'seed {s}:  '
    for r in KEEP_RATIOS:
        v = all_train_results.get((s, int(r*100)), float('nan'))
        row += f'{v:.4f}   '
    print(row)

In [ ]:
# ── Cell 7 — AUC table: mean ± std across 3 seeds ────────────────────────────
#
# Loads all 9 CA checkpoints + all 9 standard checkpoints (from nb05).
# Reports mean ± std per label per keep ratio for both variants.
# Saves ca_vs_standard_auc.csv with mean values (backward compatible with nb09).

from collections import defaultdict

RELIABLE_MIN = 5
val_pos      = df_valid[LABEL_COLS].sum().to_dict()

# Best baseline for reference
best_baseline_seed = max(
    BASELINE_SEEDS,
    key=lambda s: torch.load(CKPT_DIR / f'baseline_seed{s}_best.pt',
                             map_location='cpu', weights_only=False)['val_macro_auc'])
baseline_aucs = torch.load(
    CKPT_DIR / f'baseline_seed{best_baseline_seed}_best.pt',
    map_location='cpu', weights_only=False)['val_auc_per_label']

auc_std = defaultdict(lambda: defaultdict(list))  # [pct][label] = [seed42, ...]
auc_ca  = defaultdict(lambda: defaultdict(list))

for baseline_seed in BASELINE_SEEDS:
    for keep_ratio in KEEP_RATIOS:
        pct = int(keep_ratio * 100)
        for variant, d_dict, prefix in [
            ('std', auc_std, f'dynvit_{pct}pct_seed{baseline_seed}_best.pt'),
            ('ca',  auc_ca,  f'dynvit_ca_{pct}pct_seed{baseline_seed}_best.pt'),
        ]:
            path = CKPT_DIR / prefix
            if not path.exists():
                print(f'  [missing] {path.name}')
                continue
            d = torch.load(path, map_location='cpu', weights_only=False)
            for label, v in d['val_auc_per_label'].items():
                d_dict[pct][label].append(v)

# Print table
reliable = [l for l in LABEL_COLS if int(val_pos.get(l, 0)) >= RELIABLE_MIN]
print(f'{"Label":<33} {"base":>6}', end='')
for r in KEEP_RATIOS:
    pct = int(r*100)
    print(f'  {"std_"+str(pct)+"%":>16}  {"ca_"+str(pct)+"%":>16}', end='')
print()

for label in reliable:
    base = baseline_aucs.get(label, float('nan'))
    rare = '★' if label in RARE_LABELS else ''
    print(f'{label+rare:<33} {base:>6.4f}', end='')
    for r in KEEP_RATIOS:
        pct = int(r*100)
        sv  = auc_std[pct].get(label, [])
        cv  = auc_ca[pct].get(label, [])
        sm  = f'{np.mean(sv):.4f}±{np.std(sv):.4f}' if sv else 'n/a'
        cm  = f'{np.mean(cv):.4f}±{np.std(cv):.4f}' if cv else 'n/a'
        print(f'  {sm:>16}  {cm:>16}', end='')
    print()

# Macro AUC
print('\nMacro AUC (reliable labels):')
print(f'  baseline:  {np.nanmean([baseline_aucs.get(l,float("nan")) for l in reliable]):.4f}')
for r in KEEP_RATIOS:
    pct = int(r*100)
    for name, d_dict in [('std', auc_std), ('ca', auc_ca)]:
        macro_per_seed = [
            np.nanmean([d_dict[pct].get(l, [float('nan')])[s_idx]
                        for l in reliable if s_idx < len(d_dict[pct].get(l,[]))])
            for s_idx in range(len(BASELINE_SEEDS))
        ]
        macro_per_seed = [m for m in macro_per_seed if not np.isnan(m)]
        if macro_per_seed:
            print(f'  {name}_{pct}%: {np.mean(macro_per_seed):.4f} ± {np.std(macro_per_seed):.4f}')

# Save backward-compatible CSV (mean values, same column names as before)
csv_rows = []
for label in LABEL_COLS:
    row = {'Label': label, 'Val+': int(val_pos.get(label,0)),
           'Rare': '★' if label in RARE_LABELS else '',
           'baseline': baseline_aucs.get(label, float('nan'))}
    for r in KEEP_RATIOS:
        pct = int(r*100)
        sv  = auc_std[pct].get(label, [])
        cv  = auc_ca[pct].get(label, [])
        row[f'std_{pct}']     = np.mean(sv) if sv else float('nan')
        row[f'std_{pct}_std'] = np.std(sv)  if sv else float('nan')
        row[f'ca_{pct}']      = np.mean(cv) if cv else float('nan')
        row[f'ca_{pct}_std']  = np.std(cv)  if cv else float('nan')
    csv_rows.append(row)

pd.DataFrame(csv_rows).set_index('Label').to_csv(CKPT_DIR / 'ca_vs_standard_auc.csv')
print('\nSaved ca_vs_standard_auc.csv (mean values + std, backward compatible)')

In [ ]:
# ── Cell 8 — Extract token masks (per seed + mean) ───────────────────────────
# Per seed: masks_ca_{pct}pct_seed{s}.npy
# Mean:     masks_ca_{pct}pct.npy  (backward compatible with notebook 09)

token_masks_ca = {}

for keep_ratio in KEEP_RATIOS:
    pct       = int(keep_ratio * 100)
    seed_arrs = []

    for baseline_seed in BASELINE_SEEDS:
        path = CKPT_DIR / f'dynvit_ca_{pct}pct_seed{baseline_seed}_best.pt'
        if not path.exists():
            print(f'  [missing] {path.name} — skipping')
            continue

        ckpt_d     = torch.load(path, map_location='cpu', weights_only=False)
        base_model = timm.create_model(ckpt_d['hparams']['model'],
                                       pretrained=False, num_classes=14)
        model_ca   = DynamicViT(base_model, final_keep_ratio=keep_ratio).to(DEVICE)
        model_ca.load_state_dict(ckpt_d['model_state_dict'])
        model_ca.eval()

        masks = []
        for imgs, *_ in tqdm(val_loader_1,
                              desc=f'CA masks [{pct}% seed{baseline_seed}]',
                              leave=False):
            mask = model_ca.get_token_mask(imgs.to(DEVICE))
            masks.append(mask.cpu().numpy())

        arr = np.concatenate(masks, axis=0)
        seed_arrs.append(arr)

        np.save(MAPS_DIR / 'token_masks' / f'masks_ca_{pct}pct_seed{baseline_seed}.npy', arr)
        print(f'  [CA {pct}% seed{baseline_seed}] avg kept: '
              f'{arr.sum(axis=(1,2)).mean():.1f}/196')

    if seed_arrs:
        mean_mask = np.mean(seed_arrs, axis=0)
        np.save(MAPS_DIR / 'token_masks' / f'masks_ca_{pct}pct.npy', mean_mask)
        token_masks_ca[pct] = mean_mask
        print(f'Saved masks_ca_{pct}pct.npy — mean across {len(seed_arrs)} seeds')

In [ ]:
# ── Cell 9 — Load baseline Grad-CAM (regenerate if absent) ───────────────────
#
# These are the same maps as in notebook 05 Cell 9 — no need to recompute
# if that notebook has already run.

baseline_gcam_path = MAPS_DIR / 'gradcam' / 'gradcam_baseline.npy'

if baseline_gcam_path.exists():
    baseline_gcam = np.load(baseline_gcam_path)   # [202, 14, 14, 14]
    print(f'Loaded baseline Grad-CAM: {baseline_gcam.shape}')
else:
    print('Baseline Grad-CAM not found — regenerating...')

    def reshape_transform(tensor, h=14, w=14):
        return tensor[:, 1:, :].reshape(tensor.size(0), h, w, -1).permute(0, 3, 1, 2)

    _regen_seed = max(
        BASELINE_SEEDS,
        key=lambda s: torch.load(CKPT_DIR / f'baseline_seed{s}_best.pt',
                                 map_location='cpu', weights_only=False)['val_macro_auc'])
    _regen_d = torch.load(CKPT_DIR / f'baseline_seed{_regen_seed}_best.pt',
                           map_location='cpu', weights_only=False)
    base_model = timm.create_model(_regen_d['hparams']['model'],
                                   pretrained=False, num_classes=14)
    base_model.load_state_dict(_regen_d['model_state_dict'])
    base_model.eval().to(DEVICE)

    cam = GradCAM(model=base_model, target_layers=[base_model.blocks[-1].norm1],
                  reshape_transform=reshape_transform)

    N_VAL = len(df_valid)
    baseline_gcam = np.zeros((N_VAL, len(LABEL_COLS), 14, 14), dtype=np.float32)

    for img_idx, (imgs, *_) in enumerate(tqdm(val_loader_1, desc='Baseline Grad-CAM')):
        imgs = imgs.to(DEVICE)
        for li in range(len(LABEL_COLS)):
            cam_224 = cam(input_tensor=imgs, targets=[ClassifierOutputTarget(li)])
            baseline_gcam[img_idx, li] = cv2.resize(cam_224[0], (14, 14),
                                                     interpolation=cv2.INTER_AREA)

    np.save(baseline_gcam_path, baseline_gcam)
    print(f'Saved baseline Grad-CAM: {baseline_gcam.shape}')

In [ ]:
# ── Cell 10 — IoU: CA token masks vs baseline Grad-CAM ───────────────────────
#
# Same IoU computation as notebook 05 Cell 10 but for the CA models.
# Also loads standard DynamicViT masks from notebook 05 for direct comparison.

is_pos = df_valid[LABEL_COLS].values == 1.0   # [202, 14]
RELIABLE_MIN = 5

def threshold_gcam(cam_14, keep_ratio):
    thresh = np.percentile(cam_14, (1 - keep_ratio) * 100)
    return (cam_14 >= thresh).astype(np.float32)

def iou(a, b):
    return (a * b).sum() / (np.clip(a + b, 0, 1).sum() + 1e-8)


def compute_iou_rows(token_masks_dict, variant_name):
    rows = []
    for keep_ratio in KEEP_RATIOS:
        pct   = int(keep_ratio * 100)
        masks = token_masks_dict[pct]
        for li, label in enumerate(LABEL_COLS):
            pos_idx = np.where(is_pos[:, li])[0]
            if len(pos_idx) < RELIABLE_MIN:
                continue
            ious = [iou(masks[i], threshold_gcam(baseline_gcam[i, li], keep_ratio))
                    for i in pos_idx]
            rows.append({
                'variant':    variant_name,
                'keep_ratio': pct,
                'label':      label,
                'mean_iou':   np.mean(ious),
                'std_iou':    np.std(ious),
                'n_images':   len(ious),
                'rare':       label in RARE_LABELS,
            })
    return rows


# Load standard DynamicViT masks from notebook 05 (if available)
token_masks_std = {}
for keep_ratio in KEEP_RATIOS:
    pct  = int(keep_ratio * 100)
    npy  = MAPS_DIR / 'token_masks' / f'masks_{pct}pct.npy'
    if npy.exists():
        token_masks_std[pct] = np.load(npy)
    else:
        print(f'  [warn] masks_{pct}pct.npy not found — run notebook 05 first')

iou_rows_ca  = compute_iou_rows(token_masks_ca,  'ca')
iou_rows_std = compute_iou_rows(token_masks_std, 'std') if token_masks_std else []

df_iou_all = pd.DataFrame(iou_rows_ca + iou_rows_std)
df_iou_all.to_csv(CKPT_DIR / 'ca_vs_standard_iou.csv', index=False)
print('Saved ca_vs_standard_iou.csv')

# Summary: mean IoU per variant × keep_ratio
print('\nMean IoU (all reliable labels):')
pivot = df_iou_all.groupby(['variant', 'keep_ratio'])['mean_iou'].mean().unstack('keep_ratio')
print(pivot.round(4).to_string())

print('\nMean IoU — RARE labels only:')
pivot_rare = df_iou_all[df_iou_all['rare']].groupby(
    ['variant', 'keep_ratio'])['mean_iou'].mean().unstack('keep_ratio')
print(pivot_rare.round(4).to_string())

In [ ]:
# ── Cell 11 — Per-subgroup IoU: CA vs standard DynamicViT ────────────────────
#
# Checks whether the CA model closes the subgroup gap in spatial alignment.

from src.dataset import age_group

df_valid['age_grp'] = df_valid['Age'].apply(age_group)

SUBGROUP_COLS = [
    ('Sex',     ['Male', 'Female']),
    ('age_grp', ['<40', '40-60', '>60']),
]

def compute_subgroup_iou(token_masks_dict, variant_name):
    rows = []
    for keep_ratio in KEEP_RATIOS:
        pct   = int(keep_ratio * 100)
        masks = token_masks_dict.get(pct)
        if masks is None:
            continue
        for sg_col, sg_vals in SUBGROUP_COLS:
            for sg_val in sg_vals:
                sg_idx = df_valid.index[df_valid[sg_col] == sg_val].tolist()
                for li, label in enumerate(LABEL_COLS):
                    pos_sg = [i for i in sg_idx if is_pos[i, li]]
                    if len(pos_sg) < 3:
                        continue
                    ious = [iou(masks[i], threshold_gcam(baseline_gcam[i, li], keep_ratio))
                            for i in pos_sg]
                    rows.append({
                        'variant':    variant_name,
                        'keep_ratio': pct,
                        'subgroup':   f'{sg_col}={sg_val}',
                        'label':      label,
                        'mean_iou':   np.mean(ious),
                        'n_images':   len(ious),
                        'rare':       label in RARE_LABELS,
                    })
    return rows

sg_rows = compute_subgroup_iou(token_masks_ca, 'ca') + \
          compute_subgroup_iou(token_masks_std, 'std')
df_sg = pd.DataFrame(sg_rows)

print('Mean IoU per subgroup (collapsed over labels):')
summary = df_sg.groupby(['variant', 'keep_ratio', 'subgroup'])['mean_iou'].mean()
print(summary.unstack('subgroup').round(4).to_string())

print('\nMean IoU per subgroup — RARE labels only:')
summary_rare = df_sg[df_sg['rare']].groupby(
    ['variant', 'keep_ratio', 'subgroup'])['mean_iou'].mean()
print(summary_rare.unstack('subgroup').round(4).to_string())

df_sg.to_csv(CKPT_DIR / 'ca_subgroup_iou.csv', index=False)
print('\nSaved ca_subgroup_iou.csv')

In [ ]:
# ── Cell 12 — Three-way summary: head pruning vs DynamicViT vs CA-DynamicViT ──
#
# Loads AUC CSVs saved by notebooks 04 and 05 and combines with CA results.
# Produces a single thesis table: per-label AUC across all methods.

head_auc_csv  = CKPT_DIR / 'pruning_auc_comparison.csv'
token_auc_csv = CKPT_DIR / 'token_pruning_auc.csv'

print('PER-LABEL AUC — RARE PATHOLOGIES')
print('=' * 80)

summary_rows = []

for label in LABEL_COLS:
    n_pos = int(val_pos.get(label, 0))
    row = {'Label': label, 'Val+': n_pos, 'Rare': '★' if label in RARE_LABELS else ''}

    # Baseline (best seed, from Cell 7)
    row['baseline'] = baseline_aucs.get(label, float('nan'))

    # Head pruning (from notebook 04 CSV if available)
    if head_auc_csv.exists():
        df_head = pd.read_csv(head_auc_csv, index_col='Label')
        for col in ['25pct', '50pct', '75pct']:
            row[f'head_{col}'] = float(df_head.loc[label, col]) \
                if label in df_head.index and col in df_head.columns else float('nan')
    else:
        print('[warn] pruning_auc_comparison.csv not found — run notebook 04 first')

    # Standard DynamicViT (from notebook 05 CSV if available)
    if token_auc_csv.exists():
        df_tok = pd.read_csv(token_auc_csv, index_col='Label')
        for r in KEEP_RATIOS:
            p   = int(r * 100)
            col = f'dynvit_{p}pct'
            row[f'std_{p}'] = float(df_tok.loc[label, col]) \
                if label in df_tok.index and col in df_tok.columns else float('nan')
    else:
        print('[warn] token_pruning_auc.csv not found — run notebook 05 first')

    # CA-DynamicViT (this notebook) — mean across seeds
    for r in KEEP_RATIOS:
        p    = int(r * 100)
        vals = auc_ca[p].get(label, [])
        row[f'ca_{p}'] = np.mean(vals) if vals else float('nan')

    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index('Label')

rare_cols = ['Val+', 'Rare', 'baseline',
             'head_25pct', 'head_50pct', 'head_75pct',
             'std_70', 'std_50', 'std_30',
             'ca_70',  'ca_50',  'ca_30']
rare_cols = [c for c in rare_cols if c in df_summary.columns]

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    print(df_summary[df_summary['Rare'] == '★'][rare_cols].round(4).to_string())

df_summary[rare_cols].to_csv(CKPT_DIR / 'three_way_summary.csv')
print('\nFull table saved: three_way_summary.csv')

In [ ]:
# ── Cell 13 — Summary plots ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

pcts = [int(r * 100) for r in KEEP_RATIOS]
reliable_labels = [l for l in LABEL_COLS if int(val_pos.get(l, 0)) >= RELIABLE_MIN]

# Baseline macro AUC for reference line (mean across seeds for consistency)
best_auc = np.nanmean([baseline_aucs.get(l, float('nan')) for l in reliable_labels])

# Left: macro AUC — standard vs CA across keep ratios
ax = axes[0]

def macro_mean(variant_dict):
    """Mean macro AUC (across seeds) per keep ratio."""
    return [np.nanmean([np.mean(variant_dict[p].get(l, [float('nan')]))
                        for l in reliable_labels])
            for p in pcts]

ax.axhline(best_auc, linestyle='--', color='gray', label='Baseline')
ax.plot(pcts, macro_mean(auc_std), 'o-',  color='steelblue', label='Standard DynamicViT')
ax.plot(pcts, macro_mean(auc_ca),  's--', color='coral',     label='CA-DynamicViT')
ax.set_xlabel('Keep ratio (%)')
ax.set_ylabel('Macro AUC (reliable labels)')
ax.set_title('Macro AUC vs keep ratio')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Centre: rare-label macro AUC — standard vs CA
ax = axes[1]

def rare_macro_mean(variant_dict):
    return [np.nanmean([np.mean(variant_dict[p].get(l, [float('nan')]))
                        for l in RARE_LABELS if int(val_pos.get(l, 0)) >= RELIABLE_MIN])
            for p in pcts]

base_rare = np.nanmean([baseline_aucs.get(l, float('nan'))
                        for l in RARE_LABELS if int(val_pos.get(l, 0)) >= RELIABLE_MIN])
ax.axhline(base_rare, linestyle='--', color='gray', label='Baseline')
ax.plot(pcts, rare_macro_mean(auc_std), 'o-',  color='steelblue', label='Standard DynamicViT')
ax.plot(pcts, rare_macro_mean(auc_ca),  's--', color='coral',     label='CA-DynamicViT')
ax.set_xlabel('Keep ratio (%)')
ax.set_ylabel('Macro AUC (rare labels only)')
ax.set_title('Rare-class AUC vs keep ratio')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Right: mean IoU — standard vs CA, overall and rare labels, at 50% keep
ax = axes[2]
if not df_iou_all.empty:
    groups = ['All labels', 'Rare labels']
    x      = np.arange(len(groups))
    w      = 0.3

    def bar_iou(variant, rare_only):
        sub = df_iou_all[(df_iou_all['variant'] == variant) &
                         (df_iou_all['keep_ratio'] == 50)]
        if rare_only:
            sub = sub[sub['rare']]
        return sub['mean_iou'].mean() if not sub.empty else float('nan')

    vals_std = [bar_iou('std', False), bar_iou('std', True)]
    vals_ca  = [bar_iou('ca',  False), bar_iou('ca',  True)]

    ax.bar(x - w/2, vals_std, w, label='Standard DynamicViT', color='steelblue', alpha=0.8)
    ax.bar(x + w/2, vals_ca,  w, label='CA-DynamicViT',       color='coral',     alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(groups)
    ax.set_ylabel('Mean IoU vs Grad-CAM')
    ax.set_title('Token-mask / Grad-CAM alignment at 50% keep')
    ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(MAPS_DIR / 'ca_token_pruning_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {MAPS_DIR / "ca_token_pruning_summary.png"}')